In [ ]:
# Cell 1 — path setup (run from repo root in both VSCode and Colab)
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # if running from notebooks/ locally
# in Colab, since you %cd into the repo root first, use: sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sp_signal
from src.channel import SignalConfig, INTERFERENCE_GENERATORS

cfg = SignalConfig()
rng = np.random.default_rng(42)
print("NSPS:", cfg.NSPS)  # should print 8

In [ ]:
# Cell 2 — reproduce Figure 1: I/Q + STFT magnitude for each interference type
n_samples = 512

fig, axes = plt.subplots(len(INTERFERENCE_GENERATORS), 3, figsize=(12, 10))

for row, (name, gen) in enumerate(INTERFERENCE_GENERATORS.items()):
    z = gen(cfg, n_samples, rng)
    t = np.arange(n_samples) / cfg.Fs

    axes[row, 0].plot(t, z.real)
    axes[row, 0].set_title(f"{name} — Real")
    axes[row, 1].plot(t, z.imag)
    axes[row, 1].set_title(f"{name} — Imag")

    f, tt, Zxx = sp_signal.stft(z, fs=cfg.Fs, nperseg=64)
    axes[row, 2].pcolormesh(tt, f, np.abs(Zxx), shading='gouraud')
    axes[row, 2].set_title(f"{name} — STFT magnitude")

plt.tight_layout()
plt.savefig("../figures/fig1_reproduction.png", dpi=150)
plt.show()

In [ ]:
# Cell 3 — sanity check the full mixture pipeline (Eq. 4-6)
from src.channel import mix_signal

sample = mix_signal(cfg, n_symbols=256, rng=rng, interference_type="chirp",
                     contamination=1.0, apply_shift=True)

print("x shape:", sample["x"].shape)
print("label (interference present):", sample["label"])
print("SIR (dB):", sample["sir_db"], " SNR (dB):", sample["snr_db"])
print("Power(x):", np.mean(np.abs(sample["x"])**2))  # sanity: should be roughly O(1-3), not 0 or huge

plt.figure(figsize=(10,3))
plt.plot(sample["x"].real, label="received (real)")
plt.plot(sample["s"].real, label="clean SOI (real)", alpha=0.6)
plt.legend()
plt.title("Mixture sanity check")
plt.show()